[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dennymarcels/dubbsystem/blob/main/notebooks/colab_dubb_demo.ipynb)

# DubbSystem Colab Demo
This notebook installs DubbSystem from git, uploads an MP4, runs the dubbing pipeline, and previews the dubbed result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Before You Run
Update the repository URL in the next cell before executing the notebook. A GPU runtime is strongly recommended for transcription, large-model translation, and voice cloning.
If you have reviewed and accepted the applicable Coqui XTTS terms, set `COQUI_TOS_AGREED = "1"` in the configuration cell to skip the interactive license prompt during model download.
The notebook defaults to American English dubbing with `TARGET_LANGUAGE = "en-us"` and a larger NLLB translation model.

In [ ]:
import sys
REPO_URL = "https://github.com/dennymarcels/dubbsystem.git"
TARGET_LANGUAGE = "en-us"
TRANSCRIPTION_MODEL = "large-v3"
TRANSLATION_MODEL = "facebook/nllb-200-3.3B"
VOICE_SAMPLE_SECONDS = 60
LOG_LEVEL = "INFO"
COQUI_TOS_AGREED = "1"  # Set to "1" only after you review and accept the applicable Coqui terms.
PYTHON_BIN = "python3.11" if sys.version_info >= (3, 12) else "python3"
print(f"Notebook runtime: {sys.version.split()[0]}")
print(f"Installer Python: {PYTHON_BIN}")
print(f"Target language: {TARGET_LANGUAGE}")
print(f"Transcription model: {TRANSCRIPTION_MODEL}")
print(f"Translation model: {TRANSLATION_MODEL}")
print(f"Voice sample seconds: {VOICE_SAMPLE_SECONDS}")

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg python3.11 python3.11-venv
!git clone "$REPO_URL" /content/DubbSystem
%cd /content/DubbSystem
!rm -rf .venv
!$PYTHON_BIN -m venv .venv
!./.venv/bin/python -m pip install --upgrade pip "setuptools<81" wheel
!./.venv/bin/python -m pip install -e .
!./.venv/bin/python -m pip install "setuptools<81"

## Upload an MP4
Use the file picker to upload one source MP4 video to the Colab runtime.

In [ ]:
from google.colab import files

In [ ]:
input_path = "/content/drive/MyDrive/Cursos IA Expert/MLOps/vídeos/1.1 WSL.mp4"
print(f"Uploaded: {input_path}")

In [ ]:
from pathlib import Path
input_file = Path(input_path)
output_file = input_file.with_name(f"{input_file.stem}_dubbed{input_file.suffix}")
print(f"Expected output: {output_file}")

## Whole Process Reference
Keep this cell commented out. It shows the packaged end-to-end CLI invocation, but the staged workflow below is the recommended path when you want to inspect files after each phase.

In [ ]:
# Uncomment this command only if you want to run the packaged CLI end to end.
# The step-by-step workflow below writes the same intermediate artifacts in a way that is easier to inspect.
# !COQUI_TOS_AGREED={COQUI_TOS_AGREED} ./.venv/bin/dubb "{input_path}" "{output_file}" --target-language "{TARGET_LANGUAGE}" --transcription-model "{TRANSCRIPTION_MODEL}" --translation-model "{TRANSLATION_MODEL}" --voice-sample-seconds {VOICE_SAMPLE_SECONDS} --log-level "{LOG_LEVEL}" --keep-temp

## Step-by-Step Workflow
Run the cells below in order. Each phase writes files into the working directory so you can inspect audio, JSON artifacts, and aligned synthesis outputs before continuing.

In [ ]:
import json
from IPython.display import Audio, Markdown, Video, display
from dubb.cli import configure_logging
from dubb.pipeline import DubbingPipeline
from dubb.schemas import DubbingConfig

configure_logging(LOG_LEVEL)
config = DubbingConfig(
    input_path=input_file,
    output_path=output_file,
    target_language=TARGET_LANGUAGE,
    transcription_model=TRANSCRIPTION_MODEL,
    translation_model=TRANSLATION_MODEL,
    voice_sample_seconds=VOICE_SAMPLE_SECONDS,
)
pipeline = DubbingPipeline(config)
work_dir = pipeline.prepare_workspace()
print(f"Artifacts will be written to: {work_dir}")

## Step 1 and 2: Extract Audio and Create Speaker Sample
This creates the extracted source WAV and the speaker reference WAV. Both files remain in the working directory for listening and inspection.

In [ ]:
source_audio = pipeline.extract_source_audio()
speaker_sample = pipeline.create_speaker_sample(source_audio)
print(f"Extracted audio: {source_audio}")
print(f"Speaker sample: {speaker_sample}")
display(Markdown("### Source Audio"))
display(Audio(str(source_audio)))
display(Markdown("### Speaker Sample"))
display(Audio(str(speaker_sample)))

## Step 3, 4, and 5: Transcribe, Translate, and Build Synthesis Chunks
This produces the raw transcript JSON, the translated transcript JSON, and the merged synthesis chunk manifest used for pacing.

In [ ]:
transcript = pipeline.transcribe_source_audio(source_audio)
translated_segments = pipeline.translate_segments(transcript.segments, transcript.source_language)
synthesis_chunks = pipeline.prepare_synthesis_chunks(translated_segments)
transcript_source_path = work_dir / "transcript.source.json"
transcript_translated_path = work_dir / "transcript.translated.json"
chunks_path = work_dir / "synthesis_chunks.json"
transcript_source_payload = json.loads(transcript_source_path.read_text(encoding="utf-8"))
transcript_translated_payload = json.loads(transcript_translated_path.read_text(encoding="utf-8"))
chunks_payload = json.loads(chunks_path.read_text(encoding="utf-8"))
print(f"Detected source language: {transcript.source_language}")
print(f"Raw transcript artifact: {transcript_source_path}")
print(f"Translated transcript artifact: {transcript_translated_path}")
print(f"Synthesis chunks artifact: {chunks_path}")
display(Markdown("### Raw Transcript Sample"))
print(json.dumps(transcript_source_payload["segments"][:3], indent=2, ensure_ascii=False))
display(Markdown("### Translated Transcript Sample"))
print(json.dumps(transcript_translated_payload["segments"][:3], indent=2, ensure_ascii=False))
display(Markdown("### Synthesis Chunk Sample"))
print(json.dumps(chunks_payload[:3], indent=2, ensure_ascii=False))

## Step 6, 7, and 8: Synthesize, Compose, and Mux
This generates aligned segment WAVs, the final dubbed track WAV, and the final MP4. The synthesis manifest lists every raw and aligned chunk file.

In [ ]:
synthesized_segments = pipeline.synthesize_chunks(synthesis_chunks, speaker_sample)
dubbed_audio = pipeline.compose_dubbed_audio(synthesized_segments)
result_video = pipeline.mux_dubbed_video(dubbed_audio)
manifest_path = work_dir / "synthesis_manifest.json"
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"Synthesis manifest: {manifest_path}")
print(f"Dubbed track: {dubbed_audio}")
print(f"Dubbed video: {result_video}")
output_file = result_video
display(Markdown("### Synthesis Manifest Sample"))
print(json.dumps(manifest_payload[:3], indent=2, ensure_ascii=False))
display(Markdown("### Dubbed Audio"))
display(Audio(str(dubbed_audio)))
display(Markdown("### Dubbed Video"))
display(Video(str(result_video), embed=True))

## Inspect Generated Files
This lists the files written by the staged workflow so you can inspect any artifact directly from the Colab file browser or with additional cells.

In [ ]:
artifact_files = sorted(
    path.relative_to(work_dir).as_posix()
    for path in work_dir.rglob("*")
    if path.is_file()
)
for artifact_file in artifact_files:
    print(artifact_file)

## Preview the Result
Play the dubbed MP4 inline inside Colab.

In [ ]:
from IPython.display import Video
Video(str(output_file), embed=True)

## Download the Dubbed Video
Use this if you want to save the generated MP4 back to your machine.

In [ ]:
files.download(str(output_file))